# Planck Spectroscopy Routine for JPA and LKIPA Noise Calibration
---

We ramp up the MXC Heater power until the temperature is stabilized, then keep the heater on at the same power and run the planck spec scripts of the JPA and the LKIPA

- $\sim 40$ data points from $10 mK - 200 mK$
- $\sim 20$ data points from $200 mK - 700 mK$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
import os
import h5py
import inspect
from tqdm import tqdm
import sys
import math
import presto
from presto import lockin, utils, hardware
from presto.hardware import AdcFSample, AdcMode, DacFSample, DacMode
from blueftc.BlueforsController import BlueFTController
# Reload credentials module to get latest changes
import importlib
import power_ramp as pr
importlib.reload(pr)

# Timeout for hardware communication
import requests

# BASE URL for interacting with BlueFTC
base_url = 'http://192.168.88.25:5001'

# Print JSON 
def print_JSON(r):
    r_dict = r.json()
    for key, value in r_dict.items():
        print(f"{key}: {value}")

## 1. MXC Heater Power List

In [5]:
MXC_Heater_Power_list_low_temp = 10 * np.arange(    # from 10mk - 100mk, in steps of 10 mu W
    start= 1,
    stop = 31,
    step = 1,
) 

MXC_Heater_Power_list_medium_temp = MXC_Heater_Power_list_low_temp[-1] + 50 * np.arange(    # from 100mk - 200mk, in steps of 50 mu W
    start= 1,
    stop = 15,
    step = 1,
) 

MXC_Heater_Power_list_high_temp = MXC_Heater_Power_list_medium_temp[-1] + 200 * np.arange(    # from 200mk - 700mk, in steps of 50 mu W
    start= 1,
    stop = 21,
    step = 1,
) 

MXC_Heater_Power_list = 1e-6 * np.concatenate(( # mu W
    MXC_Heater_Power_list_low_temp, 
    MXC_Heater_Power_list_medium_temp, 
    MXC_Heater_Power_list_high_temp
))

print((MXC_Heater_Power_list) * 1e6, 'muW') # print in mu W

[  10.   20.   30.   40.   50.   60.   70.   80.   90.  100.  110.  120.
  130.  140.  150.  160.  170.  180.  190.  200.  210.  220.  230.  240.
  250.  260.  270.  280.  290.  300.  350.  400.  450.  500.  550.  600.
  650.  700.  750.  800.  850.  900.  950. 1000. 1200. 1400. 1600. 1800.
 2000. 2200. 2400. 2600. 2800. 3000. 3200. 3400. 3600. 3800. 4000. 4200.
 4400. 4600. 4800. 5000.] muW


## 2. Ramp up power and capture measurements for each temperature

In [ ]:
test_list = 1e-6 * np.array([10]) # mu W

pr.heater_ramp_up(
    MXC_heater_power_list=MXC_Heater_Power_list,
    N_runs=50
)

Experiment over, turning OFF heater... 

Requesting... http://192.168.88.25:5001/heater/update 

Heater successfully turned off. 

